# Hyperparameter tuning
Website: [Hyperparameter tuning using Ray Tune — PyTorch Tutorials 2.1…](https://docs.pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)

In [ ]:
import ray
from ray import tune
from ray.tune import Checkpoint
from ray.tune.schedulers import ASHAScheduler

## Model Architecture
- This tutorial searches for the best sizes for the fully connected layers and the learning rate
- To enable this, the `Net` class exposes the layer sizes `l1` and `l2` as configurable parameters that Ray Tune can search over

In [ ]:
class Net(nn.Module):
    def __init__(self, l1=120, l2=84):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, l1)
        self.fc2 = nn.Linear(l1, l2)
        self.fc3 = nn.Linear(l2, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)  # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

## Define the Search Space
- define the hyperparameters to tune and how Ray Tune samples them
- Ray Tune offers a variety of [search space distributions](https://docs.ray.io/en/latest/tune/api/search_space.html) to suit different parameter types: `loguniform`, `uniform`, `choice`, `randint`, `grid`, and more

In [ ]:
config = {
    "l1": tune.choice([2**i for i in range(9)]),
    "l2": tune.choice([2**i for i in range(9)]),
    "lr": tune.loguniform(1e-4, 1e-1),
    "batch_size": tune.choice([2, 4, 8, 16]),
}

- The tune.choice() accepts a list of values that are uniformly sampled from
- In this example, the `l1` and `l2` parameter values are powers of 2 between 1 and 256, and the learning rate samples on a log scale between 0.0001 and 0.1
- Sampling on a log scale enables exploration across a range of magnitudes on a relative scale, rather than an absolute scale

## Training Function
- Ray Tune requires a training function that accepts a configuration dictionary and runs the main training loop
- As Ray Tune runs different trials, it updates the configuration dictionary for each trial
- Here is the full training function

In [ ]:
def train_cifar(config, data_dir=None):
    net = Net(config["l1"], config["l2"])
    device = config["device"]

    net = net.to(device)
    if torch.cuda.device_count() > 1:
        net = nn.DataParallel(net)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=config["lr"], momentum=0.9)

    # Load checkpoint if resuming training
    checkpoint = tune.get_checkpoint()
    if checkpoint:
        with checkpoint.as_directory() as checkpoint_dir:
            checkpoint_path = Path(checkpoint_dir) / "checkpoint.pt"
            checkpoint_state = torch.load(checkpoint_path)
            start_epoch = checkpoint_state["epoch"]
            net.load_state_dict(checkpoint_state["net_state_dict"])
            optimizer.load_state_dict(checkpoint_state["optimizer_state_dict"])
    else:
        start_epoch = 0

    trainset, _testset = load_data(data_dir)

    test_abs = int(len(trainset) * 0.8)
    train_subset, val_subset = random_split(
        trainset, [test_abs, len(trainset) - test_abs]
    )

    trainloader = torch.utils.data.DataLoader(
        train_subset, batch_size=int(config["batch_size"]), shuffle=True, num_workers=8
    )
    valloader = torch.utils.data.DataLoader(
        val_subset, batch_size=int(config["batch_size"]), shuffle=True, num_workers=8
    )

    for epoch in range(start_epoch, 10):  # loop over the dataset multiple times
        running_loss = 0.0
        epoch_steps = 0
        for i, data in enumerate(trainloader, 0):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item()
            epoch_steps += 1
            if i % 2000 == 1999:  # print every 2000 mini-batches
                print(
                    "[%d, %5d] loss: %.3f"
                    % (epoch + 1, i + 1, running_loss / epoch_steps)
                )
                running_loss = 0.0

        # Validation loss
        val_loss = 0.0
        val_steps = 0
        total = 0
        correct = 0
        for i, data in enumerate(valloader, 0):
            with torch.no_grad():
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = net(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                loss = criterion(outputs, labels)
                val_loss += loss.cpu().numpy()
                val_steps += 1

        # Save checkpoint and report metrics
        checkpoint_data = {
            "epoch": epoch,
            "net_state_dict": net.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }
        with tempfile.TemporaryDirectory() as checkpoint_dir:
            checkpoint_path = Path(checkpoint_dir) / "checkpoint.pt"
            torch.save(checkpoint_data, checkpoint_path)

            checkpoint = Checkpoint.from_directory(checkpoint_dir)
            tune.report(
                {"loss": val_loss / val_steps, "accuracy": correct / total},
                checkpoint=checkpoint,
            )

    print("Finished Training")

## Key Integration Points
Using hyperparameters from the configuration dictionary

- Ray Tune updates the `config` dictionary with the hyperparameters for each trial
    - In this example, the model architecture and optimizer receive the hyperparameters from the `config` dictionary

In [ ]:
net = Net(config["l1"], config["l2"])
optimizer = optim.SGD(net.parameters(), lr=config["lr"], momentum=0.9)

### Reporting metrics and saving checkpoints

- The most important integration is communicating with Ray Tune
- Ray Tune uses the validation metrics to determine the best hyperparameter configuration and to stop underperforming trials early, saving resources
- Checkpointing enables you to later load the trained models, resume hyperparameter searches, and provides fault tolerance
- It’s also required for some Ray Tune schedulers like Population Based Training that pause and resume some trials during the search
- This code from the training function loads model and optimizer state at the start if a checkpoint exists:

In [ ]:
checkpoint = tune.get_checkpoint()
if checkpoint:
    with checkpoint.as_directory() as checkpoint_dir:
        checkpoint_path = Path(checkpoint_dir) / "checkpoint.pt"
        checkpoint_state = torch.load(checkpoint_path)
        start_epoch = checkpoint_state["epoch"]
        net.load_state_dict(checkpoint_state["net_state_dict"])
        optimizer.load_state_dict(checkpoint_state["optimizer_state_dict"])

At the end of each epoch, save a checkpoint and report the validation metrics:

In [ ]:
checkpoint_data = {
    "epoch": epoch,
    "net_state_dict": net.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}
with tempfile.TemporaryDirectory() as checkpoint_dir:
    checkpoint_path = Path(checkpoint_dir) / "checkpoint.pt"
    torch.save(checkpoint_data, checkpoint_path)

    checkpoint = Checkpoint.from_directory(checkpoint_dir)
    tune.report(
        {"loss": val_loss / val_steps, "accuracy": correct / total},
        checkpoint=checkpoint,
    )

### Multi-GPU Support
- Image classifcation models can be greatly accelerated by using GPUs
- The training function supports multi-GPU training by wrapping the model in `nn.DataParallel`

In [ ]:
if torch.cuda.device_count() > 1:
    net = nn.DataParallel(net)

This training function supports training on CPUs, a single GPU, multiple GPUs, or multiple nodes without code changes. Ray Tune automatically distributes the trials across the nodes according to the available resources. Ray Tune also supports [fractional GPUs](https://docs.ray.io/en/latest/ray-core/scheduling/accelerators.html#fractional-accelerators) so that one GPU can be shared among multiple trials, provided that the models, optimizers, and data batches fit into the GPU memory.

### Validation Split
- The original CIFAR10 dataset only has train and test subsets
- This is sufficient for training a single model, however for hyperparameter tuning, a validation subset is required
The training function creates a validation subset by reserving 20% of the training subset
- The test subset is used to evaluate the best model’s generalization error after the search completes

## Evaluation Function
After finding the optimal hyperparameters, test the model on a held-out test set to estimate the generalization error

In [ ]:
def test_accuracy(net, device="cpu", data_dir=None):
    _trainset, testset = load_data(data_dir)

    testloader = torch.utils.data.DataLoader(
        testset, batch_size=4, shuffle=False, num_workers=2
    )

    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            image_batch, labels = data
            image_batch, labels = image_batch.to(device), labels.to(device)
            outputs = net(image_batch)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

## Configure and Run Ray Tune
- With training and evaluation functions defined, configure Ray Tune to run the hyperparameter search

Scheduler for early stopping

- Ray Tune provides schedulers to improve the efficiency of the hyperparameter search by detecting underperforming trials and stopping them early
- The `ASHAScheduler` uses the Asynchronous Successive Halving Algorithm (ASHA) to aggressively terminate low-performing trials:

In [ ]:
scheduler = ASHAScheduler(
    max_t=max_num_epochs,
    grace_period=1,
    reduction_factor=2,
)

- Ray Tune provides [advanced search algorithms](https://docs.ray.io/en/latest/tune/api/suggestion.html) to smartly pick the next set of hyperparameters based on previous results, instead of relying only on random or grid search
    - Examples include [Optuna](https://docs.ray.io/en/latest/tune/api/suggestion.html#optuna) and [BayesOpt](https://docs.ray.io/en/latest/tune/api/suggestion.html#bayesopt)

**Resource Allocation**

- Tell Ray Tune what resources to allocate for each trial by passing a `resources` dictionary to `tune.with_resources`:

In [ ]:
tune.with_resources(
    partial(train_cifar, data_dir=data_dir),
    resources={"cpu": cpus_per_trial, "gpu": gpus_per_trial}
)

- Ray Tune automatically manages the placement of these trials and ensures that the trials run in isolation, so you don’t need to manually assign GPUs to processes
- For example, if you are running this experiment on a cluster of 20 machines, each with 8 GPUs, you can set `gpus_per_trial = 0.5` to schedule two concurrent trials per GPU
    - This configuration runs 320 trials in parallel across the cluster

## Creating the Tuner

- The Ray Tune API is modular and composable
- Pass your configuration to the `tune.Tuner` class to create a tuner object, then run `tuner.fit()` to start training:

In [ ]:
tuner = tune.Tuner(
    tune.with_resources(
        partial(train_cifar, data_dir=data_dir),
        resources={"cpu": cpus_per_trial, "gpu": gpus_per_trial}
    ),
    tune_config=tune.TuneConfig(
        metric="loss",
        mode="min",
        scheduler=scheduler,
        num_samples=num_trials,
    ),
    param_space=config,
)
results = tuner.fit()

After training completes, retrieve the best performing trial, load its checkpoint, and evaluate on the test set.

## Putting it all Together

In [ ]:
def main(num_trials=10, max_num_epochs=10, gpus_per_trial=0, cpus_per_trial=2):
    print("Starting hyperparameter tuning.")
    ray.init(include_dashboard=False)

    data_dir = os.path.abspath("./data")
    load_data(data_dir)  # Pre-download the dataset
    device = "cuda" if torch.cuda.is_available() else "cpu"
    config = {
        "l1": tune.choice([2**i for i in range(9)]),
        "l2": tune.choice([2**i for i in range(9)]),
        "lr": tune.loguniform(1e-4, 1e-1),
        "batch_size": tune.choice([2, 4, 8, 16]),
        "device": device,
    }
    scheduler = ASHAScheduler(
        max_t=max_num_epochs,
        grace_period=1,
        reduction_factor=2,
    )

    tuner = tune.Tuner(
        tune.with_resources(
            partial(train_cifar, data_dir=data_dir),
            resources={"cpu": cpus_per_trial, "gpu": gpus_per_trial}
        ),
        tune_config=tune.TuneConfig(
            metric="loss",
            mode="min",
            scheduler=scheduler,
            num_samples=num_trials,
        ),
        param_space=config,
    )
    results = tuner.fit()

    best_result = results.get_best_result("loss", "min")
    print(f"Best trial config: {best_result.config}")
    print(f"Best trial final validation loss: {best_result.metrics['loss']}")
    print(f"Best trial final validation accuracy: {best_result.metrics['accuracy']}")

    best_trained_model = Net(best_result.config["l1"], best_result.config["l2"])
    best_trained_model = best_trained_model.to(device)
    if gpus_per_trial > 1:
        best_trained_model = nn.DataParallel(best_trained_model)

    best_checkpoint = best_result.checkpoint
    with best_checkpoint.as_directory() as checkpoint_dir:
        checkpoint_path = Path(checkpoint_dir) / "checkpoint.pt"
        best_checkpoint_data = torch.load(checkpoint_path)

        best_trained_model.load_state_dict(best_checkpoint_data["net_state_dict"])
        test_acc = test_accuracy(best_trained_model, device, data_dir)
        print(f"Best trial test set accuracy: {test_acc}")


if __name__ == "__main__":
    # Set the number of trials, epochs, and GPUs per trial here:
    main(num_trials=10, max_num_epochs=10, gpus_per_trial=1)

# Optuna

The article: https://www.geeksforgeeks.org/deep-learning/hyperparameter-tuning-with-optuna-in-pytorch/
## What is Optuna?
Key Features of Optuna:
- Pythonic Search Spaces: Define search spaces using familiar Python syntax, including conditionals and loops.
- Efficient Optimization Algorithms: Utilizes state-of-the-art algorithms for sampling hyperparameters and efficiently pruning unpromising trials.
- Parallelization: Scale studies to tens or hundreds of workers with minimal code changes.
- Visualization: Inspect optimization histories with various plotting functions

## Import Necessary Libraries
- torch: The core PyTorch library.
- torch.nn: Contains classes and functions to build neural networks.
- torch.optim: Provides optimization algorithms like Adam.
- optuna: A hyperparameter optimization library.
- torch.utils.data.DataLoader: A utility to load data in batches.
- torchvision.datasets: Contains popular datasets like MNIST.
- torchvision.transforms: Provides image transformations.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## Define a Simple Pytorch Model
- Net: A simple neural network with one hidden layer
- `__init__`: Initializes the network with a hidden layer size `hidden_size`
- forward: Defines the forward pass. It flattens the input, applies ReLU activation after the first layer and then passes it through the second layer to get predictions

In [ ]:
class Net(nn.Module):
    def __init__(self, hidden_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28*28, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## Objective function for Optuna
- objective: The function Optuna will optimize. It defines how to train the model and evaluate its performance.

**Hyperparameters:**
- `hidden_size`: Number of neurons in the hidden layer, chosen between 128 and 512.
- `learning_rate`: Learning rate for the optimizer, chosen between 1e−41e-41e−4 and 1e−11e-11e−1 on a logarithmic scale.

Data Loading: Uses the MNIST dataset with basic transformations (converting images to tensors).

Model Training: Trains the model for one epoch. The loss from the final batch is returned to Optuna.

In [ ]:
def objective(trial):
    hidden_size = trial.suggest_int('hidden_size', 128, 512)
    learning_rate = trial.suggest_float('lr', 1e-4, 1e-1, log=True)

    transform = transforms.Compose([transforms.ToTensor()])
    train_loader = DataLoader(datasets.MNIST(
        './data', train=True, download=True, transform=transform), batch_size=32, shuffle=True)

    model = Net(hidden_size)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(1):
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    return loss.item()

## Hyperparameter Optimization with Optuna
- create_study: Creates a study object where the optimization direction is set to 'minimize' (we want to minimize the loss).
- optimize: Runs the optimization process with 5 trials, calling the objective function each time with different hyperparameters.

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)
print("Best Hyperparameters:", study.best_params)